# Task 1 — Text Generation Inference (GPT-5 + Gemini 3)

Models:
- `gpt-5` (OpenAI, hosted) — https://platform.openai.com/docs/models/gpt-5
- `gemini-3-pro-preview` (Google, hosted) — https://ai.google.dev/gemini-api/docs

Prompts: `prompts.txt` (22 prompts, `ID ||| Title ||| Prompt`). Outputs: `generated_samples/Pxx.txt`.

No keys/deps needed to explore prompts: run cells top-to-bottom. API cells are guarded.

In [ ]:
# NOTE: use %pip (not !pip) so it works when pip.exe is blocked by Device Guard.
# Only needed for real API calls; parsing/demo below needs no packages.
# %pip install -q openai google-genai google-generativeai

from pathlib import Path
import os, re

def parse_prompts(path="prompts.txt"):
    out = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = [p.strip() for p in line.split("|||")]
        if len(parts) == 3:
            pid, title, prompt = parts
        elif len(parts) == 2:
            pid, prompt = parts
            title = pid
        else:
            pid = f"P{len(out)+1:02d}"
            title, prompt = pid, line
        out.append((pid, title, prompt))
    return out

prompts = parse_prompts()
print(len(prompts), "prompts loaded")
print(prompts[0])


## Example 1 — OpenAI GPT-5 (odd IDs: P01, P03, ...)


In [ ]:
import os

def gen_gpt5(prompt, model="gpt-5", max_tokens=300, temperature=0.7):
    try:
        from openai import OpenAI
    except ImportError:
        raise RuntimeError("openai not installed. Run: %pip install -q openai")
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY not set (required for GPT-5).")
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    kwargs = {"model": model, "messages": [{"role": "user", "content": prompt}],
              "max_completion_tokens": max_tokens}
    if temperature == 1.0:
        kwargs["temperature"] = temperature
    r = client.chat.completions.create(**kwargs)
    return r.choices[0].message.content.strip()

# test one (uncomment with a key set):
# print(gen_gpt5(prompts[0][2]))
print("gen_gpt5 defined (needs OPENAI_API_KEY to call)")


## Example 2 — Google Gemini 3 (even IDs: P02, P04, ...)


In [ ]:
# %pip install -q google-genai google-generativeai  # run once if needed
import os

def gen_gemini(prompt, model="gemini-3-pro-preview", max_tokens=300, temperature=0.7):
    key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
    if not key:
        raise RuntimeError("GEMINI_API_KEY (or GOOGLE_API_KEY) not set.")
    try:
        from google import genai as new_genai
        from google.genai import types as genai_types
        client = new_genai.Client(api_key=key)
        cfg = genai_types.GenerateContentConfig(max_output_tokens=max_tokens, temperature=temperature)
        r = client.models.generate_content(model=model, contents=prompt, config=cfg)
        return r.text.strip()
    except ImportError:
        pass
    import google.generativeai as genai
    genai.configure(api_key=key)
    gem = genai.GenerativeModel(model)
    r = gem.generate_content(prompt, generation_config={"max_output_tokens": max_tokens, "temperature": temperature})
    return r.text.strip()

# test one (uncomment with a key set):
# print(gen_gemini(prompts[1][2]))
print("gen_gemini defined (needs GEMINI_API_KEY to call)")


## Offline demo (no keys, no packages) — always runnable


In [ ]:
def gen_mock(prompt, pid="", title=""):
    return (f"[MOCK {pid} - {title} - offline placeholder]\n" + prompt[:200] + " ...")

for pid, title, prompt in prompts[:2]:
    print(pid, "->", gen_mock(prompt, pid, title)[:120])
print("mock demo OK")


## Batch-generate all 22 (odd=GPT-5, even=Gemini 3)


In [ ]:
outdir = Path("generated_samples")
outdir.mkdir(exist_ok=True)
for pid, title, prompt in prompts:
    m = re.search(r"(\d+)", pid)
    n = int(m.group(1)) if m else 0
    model = "gpt-5 (OpenAI GPT-5 family, hosted API)" if n % 2 == 1 else "gemini-3-pro-preview (Google Gemini 3 family, hosted API)"
    # text = gen_gpt5(prompt) if n % 2 == 1 else gen_gemini(prompt)  # uncomment with keys
    # Path(outdir / f"{pid}.txt").write_text(f"Prompt ID: {pid}\nTitle: {title}\nModel: {model}\nPrompt: {prompt}\n---\n" + text + "\n", encoding="utf-8")
    print(pid, "->", model, "(dry-run; uncomment to call API)")
